# 09 - Multi-Horizon: Du bao 4 mat hang tai H = 1, 5, 10, 15, 20, 30, 60 (data_exo_ver2.csv)

> De `CONFIG['run_all_targets'] = True` va Run All mot lan. Notebook tu chay lan luot 4 target va hien bang multihorizon tong hop o o cuoi.

Danh gia **nhieu horizon** de do **su phan ra tin hieu** (signal decay) khi du bao cang xa. **Chi in bang ket qua (MAE, RMSE, RMSE($), MAPE, SMAPE, R2 + kiem tra dat chuan) — khong ve/luu chart.**

- **Dataset:** `data/processed/data_exo_ver2.csv`.
- **Horizons:** H ∈ {1, 5, 10, 15, 20, 30, 60} ngay.
- **Tieu chuan chat luong:** MAPE < 3% (ngan han 1-5 ngay), MAPE < 5% (trung han 10-20 ngay), RMSE($) < 2.
- **Them dac trung mua vu:** `Sin(Date)`, `Cos(Date)` (chu ky nam, day-of-year).
- **Mo hinh:** SARIMA, ARIMAX, Jump-Gated ARIMAX-CatBoost, Ridge/Linear, LightGBM, LSTM, iTransformer, GUMNet-Lite, GUMNet-Ultra, PatchTST, TFT. DL/PyTorch tu dong skip neu thieu thu vien.

> ARIMA/SARIMA dung **rolling H-step** (forecast H buoc tai moi moc, khong refit). ML/DL: target = gia mat hang dang chon tai t+H.
> Chay nang: 11 mo hinh × 7 horizon. Giam `CONFIG['horizons']` / tat `run_seq_dl`, `run_nf` de chay nhanh.

> **Jump-Gated ARIMAX-CatBoost duoc mo rong cho multi-horizon:** H1 hoi 'ngay mai co nhay gia khong'; H5..H60 hoi 'trong H ngay toi co bien dong lon khong'.


In [1]:
# === Setup ===
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")   # repo models use Keras-2 API (TF 2.17 = Keras 3)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import warnings; warnings.filterwarnings("ignore")
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

TARGETS = ["MG95", "MG92", "DO 0.001%", "DO 0.05%"]

# --- Dataset + notebook self-reference -------------------------------------------------
# This notebook loads data/processed/data_exo_ver2.csv. The batch run (other targets)
# re-executes THIS notebook and inherits FORECAST_DATA_FILE so every child uses the same file.
DATA_FILENAME = os.environ.get("FORECAST_DATA_FILE", "data_exo_ver2.csv")
NB_FILENAME   = "11_multihorizon.ipynb"
DATA_TAG      = Path(DATA_FILENAME).stem            # keeps outputs per-dataset (no clobber)

CONFIG = {
    "targets":      TARGETS,
    "target":       os.environ.get("FORECAST_TARGET", TARGETS[0]),
    "run_all_targets": True,
    "horizons":     [1, 5, 10, 15, 20, 30, 60],
    "train_ratio":  0.80,
    "val_ratio":    0.10,
    "exog_cols":    ["WTI", "USD_Index", "GPR", "BRT DTD", "BRT KH", "Brent_EU_Daily", "NAPHTHA"],
    "seasonal":     5,
    "seq_len_by_h": {1: 30, 5: 30, 10: 45, 15: 45, 20: 60, 30: 60, 60: 90},
    "dl_epochs":    30,      # epochs cho LSTM/iTransformer/GUMNet
    "nf_steps":     300,     # max_steps cho PatchTST/TFT
    "run_seq_dl":   True,    # LSTM/iTransformer/GUMNet (can tensorflow)
    "run_nf":       True,    # PatchTST/TFT (can neuralforecast)
    "run_jump_gated": True, # Jump-Gated ARIMAX-CatBoost multi-horizon
    "tune_lgbm":    False,
}
if CONFIG["target"] not in TARGETS:
    raise ValueError(f"Target khong hop le: {CONFIG['target']}. Chon mot trong {TARGETS}")
print("ROOT =", ROOT)
print("Dataset:", DATA_FILENAME, "| tag:", DATA_TAG)
print("Target:", CONFIG["target"], "| Horizons:", CONFIG["horizons"])


ROOT = e:\PCDOC\xangdau\XANG_DAU_FORECAST\XANG_DAU_FORECAST
Dataset: data_exo_ver2.csv | tag: data_exo_ver2
Target: MG95 | Horizons: [1, 5, 10, 15, 20, 30, 60]


## 1. Load du lieu + Sin(Date)/Cos(Date) + (tuy chon) News

In [2]:
from src.data_loader import load_and_engineer
TARGET = CONFIG["target"]
TARGET_SLUG = TARGET.replace("%", "pct").replace(" ", "_").replace(".", "_")
TARGET_RESULT_DIR = ROOT / "results" / DATA_TAG / "by_target" / TARGET_SLUG
TARGET_RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Load the explicit dataset file (data_exo_ver2.csv unless overridden by FORECAST_DATA_FILE).
DATA_FILE = ROOT / "data" / "processed" / DATA_FILENAME
df = load_and_engineer(data_path=str(DATA_FILE), target_col=TARGET)
print("Loaded:", DATA_FILE.name)

# --- Sin(Date) & Cos(Date): chu ky mua vu theo nam (day-of-year) ---
doy = df.index.dayofyear.values
df["DOY_sin"] = np.sin(2 * np.pi * doy / 365.25)
df["DOY_cos"] = np.cos(2 * np.pi * doy / 365.25)
print("Added Sin(Date)/Cos(Date): DOY_sin, DOY_cos")

# --- News (optional) ---
news_path = Path("E:/PCDOC/xangdau/XANG_DAU_FORECAST/XANG_DAU_FORECAST/data/news/archive_2026-06-23/daily_features.csv")
news_cols = []
if news_path.exists():
    news = pd.read_csv(news_path, parse_dates=["date"]).set_index("date")
    news = news[~news.index.duplicated(keep="last")].sort_index()
    df = df.join(news, how="left"); news_cols = list(news.columns)
    df[news_cols] = df[news_cols].fillna(0.0)
    print(f"News joined: +{len(news_cols)} cot")
else:
    print("Khong co daily_features.csv -> chay khong co news.")

feature_cols = [c for c in df.columns if c != TARGET]
print("df:", df.shape, "| features:", len(feature_cols), "| range", df.index.min().date(), "->", df.index.max().date())


Loaded: data_exo_ver2.csv
Added Sin(Date)/Cos(Date): DOY_sin, DOY_cos
News joined: +16 cot
df: (4619, 64) | features: 63 | range 2008-06-12 -> 2026-05-08


## 2. Chi so + cac ham chuan bi du lieu (tabular & sequence)

In [3]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score)
from sklearn.preprocessing import RobustScaler

# --- Quality standard ------------------------------------------------------------------
# MAPE < 3%  : short-term  (H 1-5 ngay)
# MAPE < 5%  : mid-term    (H 10-20 ngay)
# RMSE < 2$  : prices are USD -> RMSE is already in dollars (RMSE($) == RMSE)
# Long-term (H 30/60): no MAPE threshold stated -> only the RMSE($) rule applies.
RMSE_USD_THRESHOLD = 2.0
def mape_threshold(H):
    if H <= 5:  return 3.0
    if H <= 20: return 5.0
    return np.nan

def reg_metrics(y_true, y_pred, name, H):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    mae  = mean_absolute_error(yt, yp)
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    mape = float(np.mean(np.abs((yt - yp) / (np.abs(yt) + 1e-8))) * 100)
    smape = float(np.mean(2.0 * np.abs(yp - yt) / (np.abs(yt) + np.abs(yp) + 1e-8)) * 100)
    thr = mape_threshold(H)
    mape_ok = bool(np.isnan(thr) or mape < thr)
    rmse_ok = bool(rmse < RMSE_USD_THRESHOLD)
    return {"Model": name, "Horizon": H, "MAE": round(mae,4), "RMSE": round(rmse,4),
            "RMSE($)": round(rmse,4),                          # RMSE in USD (prices are USD)
            "MAPE(%)": round(mape,4), "SMAPE(%)": round(smape,4), "R2": round(float(r2_score(yt,yp)),4),
            "MAPE_thr(%)": thr, "RMSE_thr($)": RMSE_USD_THRESHOLD,
            "Pass": bool(mape_ok and rmse_ok)}

def prep_tabular(H):
    w = df.copy(); w["__y"] = w[TARGET].shift(-H); w = w.dropna(subset=["__y"])
    n = len(w); ntr = int(n*CONFIG["train_ratio"]); nvl = int(n*CONFIG["val_ratio"])
    tr, vl, te = w.iloc[:ntr], w.iloc[ntr:ntr+nvl], w.iloc[ntr+nvl:]
    sx = RobustScaler().fit(tr[feature_cols])
    Xtr, Xvl, Xte = sx.transform(tr[feature_cols]), sx.transform(vl[feature_cols]), sx.transform(te[feature_cols])
    out = dict(Xtr=Xtr, Xvl=Xvl, Xte=Xte,
               ytr=tr["__y"].values, yvl=vl["__y"].values, yte=te["__y"].values,
               dir_tr=(tr["__y"].values > tr[TARGET].values).astype(int),
               dir_te=(te["__y"].values > te[TARGET].values).astype(int),
               test_dates=te.index)
    return out

def prep_sequences(H):
    from src.data_loader import make_windows
    SEQ = CONFIG["seq_len_by_h"][H]
    n_df = len(df); ntr = int(n_df*CONFIG["train_ratio"])
    sxq = RobustScaler().fit(df[feature_cols].iloc[:ntr])
    syq = RobustScaler().fit(df[[TARGET]].iloc[:ntr])
    Xall = sxq.transform(df[feature_cols].values)
    yall = syq.transform(df[[TARGET]].values).flatten()
    Xw, yw = make_windows(Xall, yall, time_steps=SEQ, horizon=H)
    nW = len(Xw); a = int(nW*0.8); b = int(nW*0.1)
    return dict(Xw_tr=Xw[:a], Xw_vl=Xw[a:a+b], Xw_te=Xw[a+b:],
                yw_tr=yw[:a], yw_vl=yw[a:a+b], yw_te=yw[a+b:],
                scaler_y=syq, SEQ=SEQ, N_FEAT=len(feature_cols))

print("Helpers ready.")

Helpers ready.


## 3. Mo hinh thong ke — rolling H-step ARIMAX & SARIMA

In [4]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
exog_all = [c for c in CONFIG["exog_cols"] if c in df.columns]

def _arima_rolling(y, exog, split, order, seas, H, maxiter=50):
    # Fit tren [0:split], forecast H-step tai moi moc (append, khong refit).
    res = SARIMAX(y[:split], exog=None if exog is None else exog[:split],
                  order=order, seasonal_order=seas,
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=maxiter)
    cur = res; N = len(y); o = split - 1; idxs, preds = [], []
    while o + H <= N - 1:
        if exog is None: fc = cur.forecast(steps=H)
        else: fc = cur.forecast(steps=H, exog=exog[o+1:o+1+H])
        preds.append(float(np.asarray(fc)[-1])); idxs.append(o + H)
        if exog is None: cur = cur.extend(y[o+1:o+2])            # extend = O(1), khong refit
        else: cur = cur.extend(y[o+1:o+2], exog=exog[o+1:o+2])
        o += 1
    return np.array(idxs), np.array(preds)

def fit_arimax(H):
    y = df[TARGET].reset_index(drop=True).astype(float)
    ex = df[exog_all].reset_index(drop=True).astype(float)
    split = int(len(y) * 0.9)
    idx, pred = _arima_rolling(y, ex, split, (2,1,2), (0,0,0,0), H)
    return y.values[idx], pred

def fit_sarima(H):
    y = df[TARGET].reset_index(drop=True).astype(float)
    split = int(len(y) * 0.9)
    idx, pred = _arima_rolling(y, None, split, (1,1,1), (1,0,1,CONFIG["seasonal"]), H)
    return y.values[idx], pred

print("ARIMA helpers ready.")

ARIMA helpers ready.


## 4. Champion Jump-Gated ARIMAX-CatBoost multi-horizon


In [5]:
# Import the module file directly: going through `src.models` runs src/models/__init__.py,
# which eagerly imports optuna/lightgbm/catboost (via baseline_lgbm) and can fail or leave a
# stale module cached -> "cannot import name 'load_champion_frame'". This avoids all of that.
import importlib.util as _ilu
_jg_path = ROOT / "src" / "models" / "jump_gated_arimax_catboost.py"
_jg_spec = _ilu.spec_from_file_location("jump_gated_arimax_catboost", _jg_path)
_jg = _ilu.module_from_spec(_jg_spec)
sys.modules[_jg_spec.name] = _jg          # register first so @dataclass can resolve annotations
_jg_spec.loader.exec_module(_jg)
JumpGatedConfig = _jg.JumpGatedConfig
load_champion_frame = _jg.load_champion_frame
run_jump_gated_arimax_catboost_horizon = _jg.run_jump_gated_arimax_catboost_horizon

jump_details = {}
# Use the SAME dataset as the rest of the notebook (load_champion_frame defaults to ver1).
champion_df = load_champion_frame(root=ROOT, data_path=DATA_FILE)
print("Champion frame:", champion_df.shape, "|", champion_df.index.min().date(), "->", champion_df.index.max().date())

# ver2 drops some exogenous columns (GPR / BRT KH / Brent_EU_Daily); keep only what's present.
jump_exog = [c for c in CONFIG["exog_cols"] if c in champion_df.columns]
print("Jump-Gated exog:", jump_exog)

def fit_jump_gated_arimax_catboost(H):
    cfg = JumpGatedConfig(
        target=TARGET,
        horizon=H,
        train_ratio=CONFIG["train_ratio"],
        val_ratio=CONFIG["val_ratio"],
        exog_cols=jump_exog,
        arimax_order=(2, 1, 2),
        # Tuned in notebook 06: smaller grid, more stable for H1-H60.
        oof_folds=3,
        oof_min_train_frac=0.60,
        jump_thresholds=[1.5, 2.5, 4.0, 6.0],
        soft_gammas=[0.5, 1.0],
        hard_cutoffs=[0.25, 0.40, 0.55],
    )
    out = run_jump_gated_arimax_catboost_horizon(
        champion_df,
        horizon=H,
        root=ROOT,
        config=cfg,
        exact_h1=True,
        progress=True,
    )
    jump_details[H] = out
    return out["y_test"], out["pred_jump_gated"]

print("Champion Jump-Gated ARIMAX-CatBoost multi-horizon helper ready.")


Champion frame: (4619, 11) | 2008-06-12 -> 2026-05-08
Jump-Gated exog: ['WTI', 'USD_Index', 'BRT DTD', 'NAPHTHA']
Champion Jump-Gated ARIMAX-CatBoost multi-horizon helper ready.


## 5. ML models - Ridge, LightGBM, Logistic (huong)

In [6]:
from sklearn.linear_model import Ridge, LogisticRegression
import lightgbm as lgb

def fit_ridge(d):
    m = Ridge(alpha=1.0).fit(d["Xtr"], d["ytr"]); return d["yte"], m.predict(d["Xte"])

def fit_lgbm(d):
    if CONFIG["tune_lgbm"]:
        from src.models.baseline_lgbm import tune_lgbm, train_lgbm
        bp = tune_lgbm(d["Xtr"], d["ytr"], d["Xvl"], d["yvl"], n_trials=30)["best_params"]
        m = train_lgbm(d["Xtr"], d["ytr"], d["Xvl"], d["yvl"], bp)
    else:
        m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, max_depth=7, num_leaves=63,
                              subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
                              random_state=42, n_jobs=-1, verbose=-1)
        m.fit(d["Xtr"], d["ytr"], eval_set=[(d["Xvl"], d["yvl"])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    return d["yte"], m.predict(d["Xte"])

def fit_logistic(d, H):
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(d["Xtr"], d["dir_tr"])
    p = clf.predict(d["Xte"])
    return {"Model": "LogisticRegression (huong)", "Horizon": H,
            "Accuracy": round(float(accuracy_score(d["dir_te"], p)),4),
            "F1": round(float(f1_score(d["dir_te"], p, zero_division=0)),4),
            "BaseUpRate": round(float(d["dir_te"].mean()),4)}
print("ML helpers ready.")

ML helpers ready.


## 6. Deep models - LSTM, iTransformer, GUMNet-Lite/Ultra (Keras); PatchTST/TFT (neuralforecast)

In [7]:
def _seq_eval(model, s):
    pred = s["scaler_y"].inverse_transform(np.asarray(model.predict(s["Xw_te"], verbose=0)).reshape(-1,1)).flatten()
    ytru = s["scaler_y"].inverse_transform(s["yw_te"].reshape(-1,1)).flatten()
    return ytru, pred

def fit_lstm(s, H):
    import tensorflow as tf
    from tensorflow.keras import layers, Model
    tf.random.set_seed(42); tf.keras.backend.clear_session()
    inp = layers.Input((s["SEQ"], s["N_FEAT"]))
    x = layers.LSTM(64, return_sequences=True)(inp); x = layers.Dropout(0.2)(x)
    x = layers.LSTM(32)(x); x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1)(x)
    m = Model(inp, out); m.compile(optimizer="adam", loss="mse")
    es = tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)
    m.fit(s["Xw_tr"], s["yw_tr"], validation_data=(s["Xw_vl"], s["yw_vl"]),
          epochs=CONFIG["dl_epochs"], batch_size=64, callbacks=[es], verbose=0)
    return _seq_eval(m, s)

def fit_itransformer(s, H):
    from src.models.hybrid_sota import train_itransformer
    m, _ = train_itransformer(s["Xw_tr"], s["yw_tr"], s["Xw_vl"], s["yw_vl"],
                              time_steps=s["SEQ"], n_features=s["N_FEAT"], horizon=H,
                              epochs=CONFIG["dl_epochs"], batch_size=64)
    return _seq_eval(m, s)

def fit_gumnet_lite(s, H):
    from src.models.hybrid_sota import train_gumnet_lite
    m, _ = train_gumnet_lite(s["Xw_tr"], s["yw_tr"], s["Xw_vl"], s["yw_vl"],
                             time_steps=s["SEQ"], n_features=s["N_FEAT"], horizon=H,
                             epochs=CONFIG["dl_epochs"], batch_size=64)
    return _seq_eval(m, s)

def fit_gumnet_ultra(s, H):
    from src.models.hybrid_sota import train_gumnet_ultra
    m, _ = train_gumnet_ultra(s["Xw_tr"], s["yw_tr"], s["Xw_vl"], s["yw_vl"],
                              time_steps=s["SEQ"], n_features=s["N_FEAT"], horizon=H,
                              epochs=CONFIG["dl_epochs"], batch_size=64)
    return _seq_eval(m, s)

def fit_neuralforecast(H):
    from neuralforecast import NeuralForecast
    from neuralforecast.models import PatchTST, TFT
    nf_exog = [c for c in CONFIG["exog_cols"] if c in df.columns]
    base = df[[TARGET] + nf_exog].copy().asfreq("B").ffill()
    long = base.reset_index(); long.columns = ["ds"] + [TARGET] + nf_exog
    long["unique_id"] = TARGET; long["y"] = long[TARGET]
    long = long[["unique_id", "ds", "y"] + nf_exog]
    n_win = min(200, max(20, int(len(df) * 0.1)))
    common = dict(h=H, input_size=CONFIG["seq_len_by_h"][H], max_steps=CONFIG["nf_steps"],
                  scaler_type="robust", hist_exog_list=nf_exog, enable_progress_bar=False)
    nf = NeuralForecast(models=[PatchTST(**common), TFT(**common)], freq="B")
    cv = nf.cross_validation(df=long, n_windows=n_win, step_size=1)
    res = {}
    for mname in ["PatchTST", "TFT"]:
        if mname in cv.columns:
            sub = cv.dropna(subset=[mname]); res[mname] = (sub["y"].values, sub[mname].values)
    return res
print("DL helpers ready.")

DL helpers ready.


## 7. Driver - chay tat ca mo hinh tren tat ca horizon

In [8]:
import time
results = []      # regression rows (co Horizon)
clf_results = []  # logistic rows

for H in CONFIG["horizons"]:
    t0 = time.time(); print("="*70); print(f"HORIZON H = {H}"); print("="*70)
    d = prep_tabular(H)

    # --- statistical ---
    for name, fn in [("ARIMAX", fit_arimax), ("SARIMA", fit_sarima)]:
        try:
            yt, yp = fn(H); results.append(reg_metrics(yt, yp, name, H)); print(" ", results[-1])
        except Exception as e:
            print(f"  {name} skipped:", repr(e))

    # --- hybrid moi: ARIMAX-CatBoost co cong nhan dien bien dong lon ---
    if CONFIG.get("run_jump_gated", True):
        try:
            yt, yp = fit_jump_gated_arimax_catboost(H)
            results.append(reg_metrics(yt, yp, "Jump-Gated ARIMAX-CatBoost", H))
            print(" ", results[-1])
        except Exception as e:
            print("  Jump-Gated ARIMAX-CatBoost skipped:", repr(e))

    # --- linear / tree ---
    for name, fn in [("Ridge (Linear)", fit_ridge), ("LightGBM", fit_lgbm)]:
        try:
            yt, yp = fn(d); results.append(reg_metrics(yt, yp, name, H)); print(" ", results[-1])
        except Exception as e:
            print(f"  {name} skipped:", repr(e))

    # # --- logistic direction ---
    # try:
    #     clf_results.append(fit_logistic(d, H)); print(" ", clf_results[-1])
    # except Exception as e:
    #     print("  Logistic skipped:", repr(e))

    # --- sequence DL (Keras) ---
    if CONFIG["run_seq_dl"]:
        try:
            s = prep_sequences(H)
            for name, fn in [("LSTM", fit_lstm), ("iTransformer", fit_itransformer),
                             ("GUMNet-Lite", fit_gumnet_lite), ("GUMNet-Ultra", fit_gumnet_ultra)]:
                try:
                    yt, yp = fn(s, H); results.append(reg_metrics(yt, yp, name, H)); print(" ", results[-1])
                except Exception as e:
                    print(f"  {name} skipped:", repr(e))
        except Exception as e:
            print("  Sequence prep skipped:", repr(e))

    # --- PatchTST / TFT (neuralforecast) ---
    if CONFIG["run_nf"]:
        try:
            for mname, (yt, yp) in fit_neuralforecast(H).items():
                results.append(reg_metrics(yt, yp, mname, H)); print(" ", results[-1])
        except Exception as e:
            print("  PatchTST/TFT skipped:", repr(e))

    print(f"  [H={H}] done in {time.time()-t0:.1f}s")

print("\nALL HORIZONS DONE. rows:", len(results))

HORIZON H = 1
  {'Model': 'ARIMAX', 'Horizon': 1, 'MAE': 0.9543, 'RMSE': 2.0175, 'RMSE($)': 2.0175, 'MAPE(%)': 0.9639, 'SMAPE(%)': 0.9608, 'R2': 0.9872, 'MAPE_thr(%)': 3.0, 'RMSE_thr($)': 2.0, 'Pass': False}
  {'Model': 'SARIMA', 'Horizon': 1, 'MAE': 1.5108, 'RMSE': 3.0496, 'RMSE($)': 3.0496, 'MAPE(%)': 1.554, 'SMAPE(%)': 1.5528, 'R2': 0.9709, 'MAPE_thr(%)': 3.0, 'RMSE_thr($)': 2.0, 'Pass': False}
OOF fold 1: 2018-02-13 -> 2020-04-08 | MAE=0.6384
OOF fold 2: 2020-04-09 -> 2022-06-01 | MAE=0.7909
OOF fold 3: 2022-06-02 -> 2024-07-24 | MAE=1.2044
Jump-Gated ARIMAX-CatBoost MAE=1.0608
  {'Model': 'Jump-Gated ARIMAX-CatBoost', 'Horizon': 1, 'MAE': 1.0608, 'RMSE': 2.7247, 'RMSE($)': 2.7247, 'MAPE(%)': 1.0239, 'SMAPE(%)': 1.0211, 'R2': 0.9767, 'MAPE_thr(%)': 3.0, 'RMSE_thr($)': 2.0, 'Pass': False}
  {'Model': 'Ridge (Linear)', 'Horizon': 1, 'MAE': 1.3958, 'RMSE': 2.8454, 'RMSE($)': 2.8454, 'MAPE(%)': 1.4263, 'SMAPE(%)': 1.4276, 'R2': 0.9746, 'MAPE_thr(%)': 3.0, 'RMSE_thr($)': 2.0, 'Pass': Fa

2026-06-24 16:02:23,988	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-06-24 16:02:24,461	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Seed set to 1


  PatchTST/TFT skipped: Exception('PatchTST does not support historical exogenous variables.')
  [H=1] done in 174.2s
HORIZON H = 5
  {'Model': 'ARIMAX', 'Horizon': 5, 'MAE': 2.1357, 'RMSE': 3.9035, 'RMSE($)': 3.9035, 'MAPE(%)': 2.1854, 'SMAPE(%)': 2.177, 'R2': 0.9526, 'MAPE_thr(%)': 3.0, 'RMSE_thr($)': 2.0, 'Pass': False}
  {'Model': 'SARIMA', 'Horizon': 5, 'MAE': 3.7698, 'RMSE': 6.9018, 'RMSE($)': 6.9018, 'MAPE(%)': 3.8635, 'SMAPE(%)': 3.9068, 'R2': 0.8518, 'MAPE_thr(%)': 3.0, 'RMSE_thr($)': 2.0, 'Pass': False}
H=5 OOF fold 1: 2018-02-15 -> 2020-04-09 | MAE=1.5393
H=5 OOF fold 2: 2020-04-13 -> 2022-06-02 | MAE=1.8937
H=5 OOF fold 3: 2022-06-03 -> 2024-07-25 | MAE=2.9434
H=5 Jump-Gated ARIMAX-CatBoost MAPE=2.2352% | MAE=2.1905
  {'Model': 'Jump-Gated ARIMAX-CatBoost', 'Horizon': 5, 'MAE': 2.1905, 'RMSE': 4.0083, 'RMSE($)': 4.0083, 'MAPE(%)': 2.2352, 'SMAPE(%)': 2.222, 'R2': 0.9497, 'MAPE_thr(%)': 3.0, 'RMSE_thr($)': 2.0, 'Pass': False}
  {'Model': 'Ridge (Linear)', 'Horizon': 5, 'MAE'

Seed set to 1


  {'Model': 'GUMNet-Ultra', 'Horizon': 5, 'MAE': 5.4584, 'RMSE': 9.7735, 'RMSE($)': 9.7735, 'MAPE(%)': 5.6176, 'SMAPE(%)': 5.6549, 'R2': 0.7023, 'MAPE_thr(%)': 3.0, 'RMSE_thr($)': 2.0, 'Pass': False}
  PatchTST/TFT skipped: Exception('PatchTST does not support historical exogenous variables.')
  [H=5] done in 157.7s
HORIZON H = 10
  {'Model': 'ARIMAX', 'Horizon': 10, 'MAE': 2.7379, 'RMSE': 4.8061, 'RMSE($)': 4.8061, 'MAPE(%)': 2.7975, 'SMAPE(%)': 2.7803, 'R2': 0.9289, 'MAPE_thr(%)': 5.0, 'RMSE_thr($)': 2.0, 'Pass': False}
  {'Model': 'SARIMA', 'Horizon': 10, 'MAE': 4.8785, 'RMSE': 10.2419, 'RMSE($)': 10.2419, 'MAPE(%)': 4.8296, 'SMAPE(%)': 5.021, 'R2': 0.6771, 'MAPE_thr(%)': 5.0, 'RMSE_thr($)': 2.0, 'Pass': False}
H=10 OOF fold 1: 2018-02-19 -> 2020-04-14 | MAE=1.9814
H=10 OOF fold 2: 2020-04-15 -> 2022-06-03 | MAE=2.8079
H=10 OOF fold 3: 2022-06-06 -> 2024-07-25 | MAE=4.2950
H=10 Jump-Gated ARIMAX-CatBoost MAPE=3.0188% | MAE=3.0470
  {'Model': 'Jump-Gated ARIMAX-CatBoost', 'Horizon': 

...

  {'Model': 'GUMNet-Ultra', 'Horizon': 20, 'MAE': 8.8756, 'RMSE': 15.6793, 'RMSE($)': 15.6793, 'MAPE(%)': 8.9295, 'SMAPE(%)': 9.2951, 'R2': 0.2415, 'MAPE_thr(%)': 5.0, 'RMSE_thr($)': 2.0, 'Pass': False}
  PatchTST/TFT skipped: Exception('PatchTST does not support historical exogenous variables.')
  [H=20] done in 233.0s
HORIZON H = 30
  {'Model': 'ARIMAX', 'Horizon': 30, 'MAE': 4.1985, 'RMSE': 6.9632, 'RMSE($)': 6.9632, 'MAPE(%)': 4.3397, 'SMAPE(%)': 4.2629, 'R2': 0.8572, 'MAPE_thr(%)': nan, 'RMSE_thr($)': 2.0, 'Pass': False}
  {'Model': 'SARIMA', 'Horizon': 30, 'MAE': 8.5796, 'RMSE': 18.1749, 'RMSE($)': 18.1749, 'MAPE(%)': 8.0477, 'SMAPE(%)': 8.8458, 'R2': 0.0272, 'MAPE_thr(%)': nan, 'RMSE_thr($)': 2.0, 'Pass': False}
H=30 OOF fold 1: 2018-03-02 -> 2020-04-22 | MAE=2.8651
H=30 OOF fold 2: 2020-04-23 -> 2022-06-09 | MAE=5.1132
H=30 OOF fold 3: 2022-06-10 -> 2024-07-29 | MAE=7.0304
H=30 Jump-Gated ARIMAX-CatBoost MAPE=5.0640% | MAE=4.9834
  {'Model': 'Jump-Gated ARIMAX-CatBoost', 'Horiz

Seed set to 1


  {'Model': 'GUMNet-Ultra', 'Horizon': 30, 'MAE': 12.3398, 'RMSE': 20.3316, 'RMSE($)': 20.3316, 'MAPE(%)': 12.5847, 'SMAPE(%)': 13.9785, 'R2': -0.2725, 'MAPE_thr(%)': nan, 'RMSE_thr($)': 2.0, 'Pass': False}
  PatchTST/TFT skipped: Exception('PatchTST does not support historical exogenous variables.')
  [H=30] done in 289.8s
HORIZON H = 60
  {'Model': 'ARIMAX', 'Horizon': 60, 'MAE': 3.7506, 'RMSE': 5.1157, 'RMSE($)': 5.1157, 'MAPE(%)': 4.1664, 'SMAPE(%)': 4.1186, 'R2': 0.928, 'MAPE_thr(%)': nan, 'RMSE_thr($)': 2.0, 'Pass': False}
  {'Model': 'SARIMA', 'Horizon': 60, 'MAE': 10.9924, 'RMSE': 22.4086, 'RMSE($)': 22.4086, 'MAPE(%)': 9.9718, 'SMAPE(%)': 11.3846, 'R2': -0.382, 'MAPE_thr(%)': nan, 'RMSE_thr($)': 2.0, 'Pass': False}
H=60 OOF fold 1: 2018-03-22 -> 2020-05-07 | MAE=3.9968
H=60 OOF fold 2: 2020-05-08 -> 2022-06-20 | MAE=7.1781
H=60 OOF fold 3: 2022-06-21 -> 2024-08-01 | MAE=7.1871
H=60 Jump-Gated ARIMAX-CatBoost MAPE=5.4410% | MAE=4.8030
  {'Model': 'Jump-Gated ARIMAX-CatBoost', '

Seed set to 1


  {'Model': 'GUMNet-Ultra', 'Horizon': 60, 'MAE': 11.8844, 'RMSE': 21.7267, 'RMSE($)': 21.7267, 'MAPE(%)': 11.2516, 'SMAPE(%)': 12.8971, 'R2': -0.4346, 'MAPE_thr(%)': nan, 'RMSE_thr($)': 2.0, 'Pass': False}
  PatchTST/TFT skipped: Exception('PatchTST does not support historical exogenous variables.')
  [H=60] done in 350.8s

ALL HORIZONS DONE. rows: 63


## 8. Ket qua - bang + signal decay theo horizon

In [9]:
res_df = pd.DataFrame(results)
res_df.insert(0, "Target", TARGET)
target_csv = TARGET_RESULT_DIR / "multihorizon_results.csv"
res_df.to_csv(target_csv, index=False)

combined_path = ROOT / "results" / DATA_TAG / "multihorizon_results_all_targets_11.csv"
if combined_path.exists():
    previous_rows = pd.read_csv(combined_path)
    previous_rows = previous_rows[previous_rows["Target"] != TARGET]
    combined_rows = pd.concat([previous_rows, res_df], ignore_index=True)
else:
    combined_rows = res_df.copy()
combined_rows.to_csv(combined_path, index=False)

if TARGET == "MG95":
    res_df.drop(columns=["Target"]).to_csv(ROOT / "results" / DATA_TAG / "multihorizon_results_11.csv", index=False)

print("=== Full results (long) ===")
display(res_df)

# Pivot: metrics theo (Model x Horizon)
for met in ["R2", "MAPE(%)", "MAE", "RMSE", "RMSE($)", "SMAPE(%)"]:
    piv = res_df.pivot_table(index="Model", columns="Horizon", values=met)
    print(f"\n--- {met} (Model x Horizon) ---"); display(piv)

# --- Quality-standard check: MAPE<3% (H1-5), MAPE<5% (H10-20), RMSE($)<2 ---
qa = res_df[["Model", "Horizon", "MAPE(%)", "MAPE_thr(%)", "RMSE($)", "RMSE_thr($)", "Pass"]].copy()
print("\n=== Quality check vs standard (MAPE<3% H1-5 | MAPE<5% H10-20 | RMSE($)<2) ===")
display(qa)
pass_piv = res_df.pivot_table(index="Model", columns="Horizon", values="Pass", aggfunc="first")
print("\n--- Pass (Model x Horizon) ---"); display(pass_piv)
print(f"Models meeting standard: {int(res_df['Pass'].sum())}/{len(res_df)}")

print(f"Saved -> {target_csv}")
print(f"Updated -> {combined_path}")


=== Full results (long) ===


,Target,Model,Horizon,MAE,RMSE,RMSE($),MAPE(%),SMAPE(%),R2,MAPE_thr(%),RMSE_thr($),Pass
0,MG95,ARIMAX,1,0.9543,2.0175,2.0175,0.9639,0.9608,0.9872,3.0,2.0,False
1,MG95,SARIMA,1,1.5108,3.0496,3.0496,1.5540,1.5528,0.9709,3.0,2.0,False
2,MG95,Jump-Gated ARIMAX-CatBoost,1,1.0608,2.7247,2.7247,1.0239,1.0211,0.9767,3.0,2.0,False
3,MG95,Ridge (Linear),1,1.3958,2.8454,2.8454,1.4263,1.4276,0.9746,3.0,2.0,False
4,MG95,LightGBM,1,1.9601,3.6035,3.6035,2.0137,2.0422,0.9592,3.0,2.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...
58,MG95,LightGBM,60,12.7022,20.8011,20.8011,12.8031,13.3254,-0.3434,NaN,2.0,False
59,MG95,LSTM,60,16.4406,28.3382,28.3382,16.0541,18.9576,-1.4406,NaN,2.0,False
60,MG95,iTransformer,60,16.6548,29.2886,29.2886,16.0514,18.0405,-1.6070,NaN,2.0,False
61,MG95,GUMNet-Lite,60,15.9289,27.5929,27.5929,15.5697,17.1971,-1.3139,NaN,2.0,False



--- R2 (Model x Horizon) ---


Horizon,1,5,10,15,20,30,60
Model,,,,,,,
ARIMAX,0.9872,0.9526,0.9289,0.8958,0.8716,0.8572,0.9280
GUMNet-Lite,0.7027,0.5791,0.3628,0.1666,-0.0550,-0.2527,-1.3139
GUMNet-Ultra,0.8074,0.7023,0.4144,0.3581,0.2415,-0.2725,-0.4346
Jump-Gated ARIMAX-CatBoost,0.9767,0.9497,0.8899,0.8648,0.7734,0.7342,0.8973
LSTM,0.8357,0.5260,0.4251,0.3669,-0.0028,-0.4418,-1.4406
LightGBM,0.9592,0.8506,0.6386,0.3791,0.1513,0.0463,-0.3434
Ridge (Linear),0.9746,0.8361,0.5516,0.1949,-0.1843,-0.2911,-0.1340
SARIMA,0.9709,0.8518,0.6771,0.4644,0.2847,0.0272,-0.3820
iTransformer,0.5494,0.3122,0.0538,0.0764,-0.2534,-0.6204,-1.6070



--- MAPE(%) (Model x Horizon) ---


...


=== Quality check vs standard (MAPE<3% H1-5 | MAPE<5% H10-20 | RMSE($)<2) ===


,Model,Horizon,MAPE(%),MAPE_thr(%),RMSE($),RMSE_thr($),Pass
0,ARIMAX,1,0.9639,3.0,2.0175,2.0,False
1,SARIMA,1,1.5540,3.0,3.0496,2.0,False
2,Jump-Gated ARIMAX-CatBoost,1,1.0239,3.0,2.7247,2.0,False
3,Ridge (Linear),1,1.4263,3.0,2.8454,2.0,False
4,LightGBM,1,2.0137,3.0,3.6035,2.0,False
...,...,...,...,...,...,...,...
58,LightGBM,60,12.8031,NaN,20.8011,2.0,False
59,LSTM,60,16.0541,NaN,28.3382,2.0,False
60,iTransformer,60,16.0514,NaN,29.2886,2.0,False
61,GUMNet-Lite,60,15.5697,NaN,27.5929,2.0,False



--- Pass (Model x Horizon) ---


Horizon,1,5,10,15,20,30,60
Model,,,,,,,
ARIMAX,False,False,False,False,False,False,False
GUMNet-Lite,False,False,False,False,False,False,False
GUMNet-Ultra,False,False,False,False,False,False,False
Jump-Gated ARIMAX-CatBoost,False,False,False,False,False,False,False
LSTM,False,False,False,False,False,False,False
LightGBM,False,False,False,False,False,False,False
Ridge (Linear),False,False,False,False,False,False,False
SARIMA,False,False,False,False,False,False,False
iTransformer,False,False,False,False,False,False,False


Models meeting standard: 0/63
Saved -> e:\PCDOC\xangdau\XANG_DAU_FORECAST\XANG_DAU_FORECAST\results\data_exo_ver2\by_target\MG95\multihorizon_results.csv
Updated -> e:\PCDOC\xangdau\XANG_DAU_FORECAST\XANG_DAU_FORECAST\results\data_exo_ver2\multihorizon_results_all_targets_11.csv


## 9. Ghi chu

- **Signal decay:** do chinh xac giam khi H tang (R2 giam, MAPE/MAE tang) — bang pivot (Model x Horizon) o tren dinh luong dieu nay cho tung mo hinh. **Khong ve chart, chi in bang.**
- **Dataset:** `data/processed/data_exo_ver2.csv` (doc tuong minh qua `DATA_FILENAME`). Ket qua luu theo tag dataset: `results/{DATA_TAG}/...`.
- **Sin(Date)/Cos(Date):** `DOY_sin/cos` (chu ky nam) bo sung mua vu; co the giup nhat o H lon (30/60).
- ARIMA/SARIMA: rolling H-step (append, khong refit) — nhanh va cong bang.
- DL Keras + PatchTST/TFT skip neu thieu thu vien; cai xong chay lai. Giam `CONFIG['horizons']` hoac tat `run_seq_dl`/`run_nf` neu muon nhanh.
- Ket qua: `results/{DATA_TAG}/multihorizon_results.csv`, `results/{DATA_TAG}/multihorizon_results_all_targets.csv`.


In [10]:
# Chay cac target con lai bang kernel phu, khong tao them file notebook.
is_batch_child = os.environ.get("FORECAST_BATCH_CHILD") == "1"
if CONFIG.get("run_all_targets", False) and not is_batch_child:
    from src.notebook_batch_runner import run_notebook_targets

    os.environ["FORECAST_DATA_FILE"] = DATA_FILENAME   # children inherit the same dataset
    remaining_targets = [t for t in CONFIG["targets"] if t != TARGET]
    run_notebook_targets(
        ROOT / "notebooks" / NB_FILENAME,
        remaining_targets,
        timeout=None,
    )

    all_results_path = ROOT / "results" / DATA_TAG / "multihorizon_results_all_targets_11.csv"
    all_results = pd.read_csv(all_results_path)
    target_order = {name: pos for pos, name in enumerate(CONFIG["targets"])}
    all_results["__target_order"] = all_results["Target"].map(target_order)
    all_results = all_results.sort_values(["__target_order", "Horizon", "MAPE(%)"]).drop(columns="__target_order")
    print("HOAN TAT 4 TARGETS - KET QUA MULTIHORIZON")
    display(all_results)


[Batch 1/3] Bat dau target: MG92
[Batch 1/3] Hoan tat MG92 sau 2243.8s
[Batch 2/3] Bat dau target: DO 0.001%
[Batch 2/3] Hoan tat DO 0.001% sau 2097.1s
[Batch 3/3] Bat dau target: DO 0.05%
[Batch 3/3] Hoan tat DO 0.05% sau 2139.8s
HOAN TAT 4 TARGETS - KET QUA MULTIHORIZON


,Target,Model,Horizon,MAE,RMSE,RMSE($),MAPE(%),SMAPE(%),R2,MAPE_thr(%),RMSE_thr($),Pass
0,MG95,ARIMAX,1,0.9543,2.0175,2.0175,0.9639,0.9608,0.9872,3.0,2.0,False
2,MG95,Jump-Gated ARIMAX-CatBoost,1,1.0608,2.7247,2.7247,1.0239,1.0211,0.9767,3.0,2.0,False
3,MG95,Ridge (Linear),1,1.3958,2.8454,2.8454,1.4263,1.4276,0.9746,3.0,2.0,False
1,MG95,SARIMA,1,1.5108,3.0496,3.0496,1.5540,1.5528,0.9709,3.0,2.0,False
4,MG95,LightGBM,1,1.9601,3.6035,3.6035,2.0137,2.0422,0.9592,3.0,2.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...
250,DO 0.05%,GUMNet-Lite,60,18.2917,41.1034,41.1034,13.3584,16.5405,-0.5825,NaN,2.0,False
247,DO 0.05%,LightGBM,60,18.0994,37.9559,37.9559,14.0273,16.2112,-0.3788,NaN,2.0,False
251,DO 0.05%,GUMNet-Ultra,60,18.9513,38.3635,38.3635,14.7594,17.6054,-0.3786,NaN,2.0,False
248,DO 0.05%,LSTM,60,23.5025,47.7837,47.7837,18.2667,23.0224,-1.1387,NaN,2.0,False
